# Week 5 · Notebook 2  Embeddings & Attention Lab

**Semantic similarity search over commodity descriptions, then scaled dot-product attention + causal masking in NumPy.**

```
# Requirements: pip install sentence-transformers numpy pandas
```

```
# ⚠️ REQUIRES: internet for model download (CPU only)
```

The `all-MiniLM-L6-v2` model downloads on first use. A deterministic offline fallback keeps the notebook runnable without internet. Part of AI Engineering Lab · ZoroLogistics case study.

## Part 1  Embeddings & similarity search

An **embedding** maps text to a dense vector so that similar meanings land near each other. We embed commodity descriptions with `all-MiniLM-L6-v2` and find, for each query, the closest commodity by cosine similarity  the primitive behind RAG retrieval (Week 7).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (assumes cwd == week-NN/)
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro").is_dir():
        sys.path.insert(0, str(_p)); break

from zoro import data
import numpy as np, pandas as pd

# Deterministic catalog of commodity descriptions (longer than the bare label).
_descriptors = {
    "electronics": "consumer electronics, laptops, and circuit boards in anti-static packaging",
    "auto parts": "automotive parts, brake assemblies, and engine components on pallets",
    "apparel": "clothing, footwear, and textile garments in cartons",
    "food & beverage": "perishable food, beverages, and dry goods requiring temperature control",
    "pharmaceuticals": "pharmaceuticals and medical supplies needing cold-chain handling",
    "construction materials": "steel beams, cement, and construction materials on flatbeds",
    "chemicals": "industrial chemicals and solvents with hazmat placards",
    "paper products": "paper rolls, cardboard, and packaging materials",
    "furniture": "flat-pack furniture, home goods, and fixtures",
    "machinery": "heavy machinery, pumps, and generators",
    "textiles": "raw textiles, fabrics, and yarn in bales",
    "perishables": "fresh produce and chilled goods on expedited lanes",
}
commodities = list(_descriptors.keys())
descriptions = [f"{c}: {_descriptors[c]}" for c in commodities]
queries = [
    "temperature-sensitive medical cargo that must stay cold",
    "hazardous solvents that need placards",
    "flat-pack home furniture on a standard lane",
]
print(len(descriptions), "commodity descriptions;", len(queries), "queries")

## Load the embedding model

`all-MiniLM-L6-v2` is a small (22M-param) sentence transformer that runs on CPU and needs internet only for the one-time download. If it cannot download, we fall back to a seeded hash-based embedding so the rest of the notebook still completes and prints a score.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    print("all-MiniLM-L6-v2 loaded, dim =", embedder.get_sentence_embedding_dimension())
except Exception as e:
    embedder = None
    print("⚠️ could not download all-MiniLM-L6-v2:", type(e).__name__, "-", e)
    print("   falling back to a deterministic hash-based embedding (offline).")

def embed(texts):
    if embedder is not None:
        vecs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        return np.asarray(vecs, dtype=float)
    rng = np.random.default_rng(42)
    dim, vocab = 64, 4096
    proj = rng.normal(0.0, 1.0, size=(dim, vocab))
    out = []
    for t in texts:
        v = np.zeros(vocab)
        for w in t.lower().split():
            v[hash(w) % vocab] += 1.0
        p = proj @ v
        n = np.linalg.norm(p)
        out.append(p / n if n > 0 else np.zeros(dim))
    return np.stack(out)

## Similarity search

Embed every description and every query, then rank descriptions by cosine similarity. This is the exact retrieval loop Week 7 turns into a RAG bot  here with no vector database, just NumPy.

In [ ]:
def cosine(a, b):
    a = a / (np.linalg.norm(a) + 1e-9)
    b = b / (np.linalg.norm(b) + 1e-9)
    return float(a @ b)

desc_vecs = embed(descriptions)
query_vecs = embed(queries)

top_sims = []
for i, q in enumerate(queries):
    sims = [cosine(query_vecs[i], dv) for dv in desc_vecs]
    order = np.argsort(sims)[::-1]
    top_sims.append(sims[order[0]])
    print(f"\nquery: {q!r}")
    for rank in range(3):
        j = order[rank]
        print(f"  #{rank+1}  sim={sims[j]:.3f}  {commodities[j]}")

## Part 2  Scaled dot-product attention in NumPy

Attention is how a token gathers information from other tokens: each token's embedding is projected into a **Query** ("what am I looking for?"), a **Key** ("what do I contain?"), and a **Value** ("what do I contribute?"). The scores are `Q·Kᵀ / √d_k`, softmaxed, and used to blend the values.

In [ ]:
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)   # set future positions to -inf
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

## Causal masking

During generation a token must not peek at the future, so a **causal mask** zeroes out every position after the current one. Each token attends only to itself and earlier tokens (a lower-triangular mask).

In [ ]:
def causal_mask(n):
    return np.tril(np.ones((n, n), dtype=bool))

n = 5
print("causal mask (1 = allowed, rows attend to columns):")
print(causal_mask(n).astype(int))

## A toy sequence, visualized

We run one attention head over a 5-token sequence with random (seeded) embeddings, print the raw weights, and render a compact ASCII heatmap. Watch the lower-triangular pattern: the first token attends only to itself, the last token attends to everything before it.

In [ ]:
rng = np.random.default_rng(7)
seq = ["shipment", "delayed", "due", "to", "storm"]
d = 8
emb = rng.normal(0, 1, size=(len(seq), d))
Q = K = V = emb  # toy: single head, identity projections

out, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask(len(seq)))

print("attention weights (query token -> key token):")
print("       " + " ".join(f"{w[:6]:>7}" for w in seq))
for i, row in enumerate(weights):
    print(f"{seq[i][:6]:>6} " + " ".join(f"{v:7.3f}" for v in row))

print("\nASCII heatmap (dark = higher attention):")
print("      " + "".join(f"{w[:4]:>6}" for w in seq))
for i, row in enumerate(weights):
    print(f"{seq[i][:4]:>5} " + "".join(("█" * int(round(v * 10))).ljust(5) + " " for v in row))

## Why this is the whole ballgame

Attention moves information *between* tokens; the KV cache (Week 5 reading, knowledge-base 03) is what makes generation affordable; and the causal mask is why models can run incrementally, one token at a time. The mental model you built here  tokens → embeddings → attention → distribution → sample  is the same one that explains every cost and latency decision in the weeks ahead.

In [ ]:
# Week 5 · Notebook 2 headline metric: top-1 cosine similarity of the best
# commodity match for the first query (temperature-sensitive medical cargo).
print("WEEK5_NB2_TOP1_SIMILARITY:", round(top_sims[0], 4))